#### 2. Validate data from API using Pydantic
Use this code snippet to get a random dad joke

In [6]:
import requests

headers = {"Accept": "application/json"}
response = requests.get("https://icanhazdadjoke.com/", headers=headers)

print(response.json())

{'id': 'WD5oWLuXgFd', 'joke': "I used to work at a stationery store.  But, I didn't feel like I was going anywhere.\r\n\r\nSo, I got a job at a travel agency.  Now, I know I'll be going places.", 'status': 200}


In [9]:
joke_dict = response.json()

#### a) Create a Pydantic model with name Joke with the following fields

- id with type integer
- joke with type string

In [19]:
from pydantic import BaseModel, ValidationError

class Joke (BaseModel):
    id: str
    joke: str 

j1 = Joke(id="1", joke="Why did the chicken cross the road? To get to the other side!")

#### b) Validate the data from the API using the Joke model. 

Test out your Joke instance to see that you can access the joke and id fields.

In [15]:
# use kwargs to unpack the dictionary into the model
j2 = Joke(**joke_dict)

In [17]:
j2.id, j2.joke

('WD5oWLuXgFd',
 "I used to work at a stationery store.  But, I didn't feel like I was going anywhere.\r\n\r\nSo, I got a job at a travel agency.  Now, I know I'll be going places.")

In [18]:
import requests
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

# --- A) Create a Pydantic model with name Joke ---

class Joke(BaseModel):
    """
    Pydantic model to validate the response structure from 
    https://icanhazdadjoke.com/
    
    The 'id' field is defined as a string because the API returns an 
    alphanumeric identifier (e.g., 'WD5oWLuXgFd').
    """
    # CORRECTED: Use str for the ID since the API returns an alphanumeric string.
    id: str = Field(description="The unique alphanumeric ID of the joke.")
    joke: str = Field(description="The full text of the joke.")
    # The 'status' field is often present but optional for our core data model
    status: Optional[int] = Field(None, description="HTTP Status code (usually 200).")


def get_and_validate_joke() -> Joke | None:
    """Fetches a random joke and validates the response using the Joke model."""
    
    API_URL = "https://icanhazdadjoke.com/"
    headers = {"Accept": "application/json"}
    
    print(f"--- Fetching joke from {API_URL} ---")
    try:
        # 1. Fetch Data
        response = requests.get(API_URL, headers=headers)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        raw_data = response.json()
        
        # 2. Validate Data using the Joke model (Task B)
        print("Raw Data Received (JSON):", raw_data)
        
        joke_instance = Joke.model_validate(raw_data)
        
        print("\n--- Data Validation Successful! ---")
        return joke_instance

    except requests.exceptions.RequestException as e:
        print(f"\n[ERROR] Network or HTTP error during request: {e}")
        return None
    except ValidationError as e:
        print(f"\n[ERROR] Pydantic Validation Failed!")
        print(e.errors())
        return None
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")
        return None


# --- Execution and Testing (Task B) ---

validated_joke = get_and_validate_joke()

if validated_joke:
    print("\n--- Testing Joke Instance Access ---")
    
    # Access the joke and id fields
    print(f"Successfully accessed ID (type: {type(validated_joke.id).__name__}): {validated_joke.id}")
    print("Joke:")
    print("-------------------------------------------------------------------")
    print(validated_joke.joke)
    print("-------------------------------------------------------------------")

    # You can also access the optional status field if it was present in the JSON
    if validated_joke.status is not None:
         print(f"Status: {validated_joke.status}")
    else:
         print("Status field was not included in the model or JSON.")

--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'LRnGeVfiNe', 'joke': 'Today a man knocked on my door and asked for a small donation towards the local swimming pool. I gave him a glass of water.', 'status': 200}

--- Data Validation Successful! ---

--- Testing Joke Instance Access ---
Successfully accessed ID (type: str): LRnGeVfiNe
Joke:
-------------------------------------------------------------------
Today a man knocked on my door and asked for a small donation towards the local swimming pool. I gave him a glass of water.
-------------------------------------------------------------------
Status: 200


#### c) Now create a new Joke Pydantic model that also have the field words_in_joke. 
- This is a computed field and a property 
- Note that computed_field is imported from pydantic. Validate a random joke with your new Joke model.

In [20]:
from pydantic import computed_field

class JokeWithWordCount(Joke):
    @computed_field
    @property
    def words_in_joke(self) -> int:
        """Computes the number of words in the joke."""
        return len(self.joke.split())
    


In [21]:
# use kwargs to unpack the dictionary into the model
jwwc = JokeWithWordCount(**joke_dict)

In [22]:
jwwc

JokeWithWordCount(id='WD5oWLuXgFd', joke="I used to work at a stationery store.  But, I didn't feel like I was going anywhere.\r\n\r\nSo, I got a job at a travel agency.  Now, I know I'll be going places.", words_in_joke=33)

#### d) Request 10 jokes from the api and validate them into many Jokes instances that you store into a list. 
Make sure to use sleep for 5 seconds to not request from the API too much.

In [25]:
def get_and_validate_joke() -> Joke | None:
    """Fetches a random joke and validates the response using the Joke model."""
    
    API_URL = "https://icanhazdadjoke.com/"
    headers = {"Accept": "application/json"}
    
    print(f"--- Fetching joke from {API_URL} ---")
    try:
        # 1. Fetch Data
        response = requests.get(API_URL, headers=headers)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        raw_data = response.json()
        
        # 2. Validate Data using the Joke model (Task B)
        print("Raw Data Received (JSON):", raw_data)
        
        joke_instance = JokeWithWordCount.model_validate(raw_data)
        
        print("\n--- Data Validation Successful! ---")
        return joke_instance

    except requests.exceptions.RequestException as e:
        print(f"\n[ERROR] Network or HTTP error during request: {e}")
        return None
    except ValidationError as e:
        print(f"\n[ERROR] Pydantic Validation Failed!")
        print(e.errors())
        return None
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")
        return None


joke_list = []
for i in range(10):
    joke = get_and_validate_joke()
    if joke:
        joke_list.append(joke)
    import time
    time.sleep(5)  # Sleep for 5 seconds to avoid hitting the API too frequently

joke_list

--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'TKmbUDI6UDd', 'joke': 'What do you call an elephant that doesn’t matter? An irrelephant.', 'status': 200}

--- Data Validation Successful! ---
--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'I6h3ozAIYDd', 'joke': 'Where do hamburgers go to dance? The meat-ball.', 'status': 200}

--- Data Validation Successful! ---
--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'OfNmyI6Ed', 'joke': 'Why did the cookie cry? It was feeling crumby.', 'status': 200}

--- Data Validation Successful! ---
--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'SKeqOucNeFd', 'joke': 'What time did the man go to the dentist? Tooth hurt-y.', 'status': 200}

--- Data Validation Successful! ---
--- Fetching joke from https://icanhazdadjoke.com/ ---
Raw Data Received (JSON): {'id': 'rHBQuXLR7h', 'joke': 'Why did the

[JokeWithWordCount(id='TKmbUDI6UDd', joke='What do you call an elephant that doesn’t matter? An irrelephant.', words_in_joke=11),
 JokeWithWordCount(id='I6h3ozAIYDd', joke='Where do hamburgers go to dance? The meat-ball.', words_in_joke=8),
 JokeWithWordCount(id='OfNmyI6Ed', joke='Why did the cookie cry? It was feeling crumby.', words_in_joke=9),
 JokeWithWordCount(id='SKeqOucNeFd', joke='What time did the man go to the dentist? Tooth hurt-y.', words_in_joke=11),
 JokeWithWordCount(id='rHBQuXLR7h', joke='Why did the cookie cry?\r\nBecause his mother was a wafer so long', words_in_joke=13),
 JokeWithWordCount(id='CtWgqjbMeib', joke='What kind of bagel can fly? A plain bagel.', words_in_joke=9),
 JokeWithWordCount(id='jyPCYTKuskb', joke='How did Darth Vader know what Luke was getting for Christmas? He felt his presents.', words_in_joke=15),
 JokeWithWordCount(id='mGlbFtk3Tvc', joke='Why did the m&m go to school? Because it wanted to be a Smartie!', words_in_joke=14),
 JokeWithWordCount(i